In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, average_precision_score, recall_score
import matplotlib as plt

In [ ]:
aberporth = np.load("aberporth.npy")
cardiff = np.load("cardiff.npy")
valley = np.load("valley.npy")

print(aberporth.shape)
print(cardiff.shape)
print(valley.shape)

print(aberporth)
print(cardiff)
print(valley)

I will select 3 models to compare. Random Forests, SVM and kNN. I predict random forests will work the best. I picked these 3 because I think they are going to work best with the low number of features - 5. Having random forest also allows us to see how a network technique compares against the other two, and kNN allows us to see how well a computationally cheaper algorithm will compare. SVM is kind of the jack of all trades here I guess. 

In [ ]:
#Preparing data
#split into X and Y
aberporthX = aberporth[:, :5]
aberporthY = aberporth[:, -1]

cardiffX = cardiff[:, :5]
cardiffY = cardiff[:, -1]

valleyX = valley[:, :5]
valleyY = valley[:, -1]

#Training and testing datasets
aTrainX, aTestX, aTrainY, aTestY = train_test_split(aberporthX, aberporthY, test_size=0.3)
cTrainX, cTestX, cTrainY, cTestY = train_test_split(cardiffX, cardiffY, test_size=0.3)
vTrainX, vTestX, vTrainY, vTestY = train_test_split(valleyX, valleyY, test_size=0.3)

#Scaling data
#KNN and SVM require data scaling since the method of regression behind them rely on distance between data points
#Therefore we will create scaled datasets for these models so that no single feature dominates the decision making
scaler = StandardScaler()
aTrainXScaled = scaler.fit_transform(aTrainX)
aTestXScaled = scaler.transform(aTestX)
cTrainXScaled = scaler.fit_transform(cTrainX)
cTestXScaled = scaler.transform(cTestX)
vTrainXScaled = scaler.fit_transform(vTrainX)
vTestXScaled = scaler.transform(vTestX)

In [ ]:
#KNN regressor
paramGridKnn = {
    "n_neighbors": range(1,51),
    "weights": ['uniform', 'distance'],
}
gridSearchKnn = GridSearchCV(estimator=KNeighborsRegressor(), param_grid=paramGridKnn, cv=5, n_jobs=-1)

#Training for Aberporth
gridSearchKnn.fit(aTrainXScaled, aTrainY)
Aknn = KNeighborsRegressor()
Aknn.set_params(**gridSearchKnn.best_params_)

#Training for Cardiff
gridSearchKnn.fit(cTrainXScaled, cTrainY)
Cknn = KNeighborsRegressor()
Cknn.set_params(**gridSearchKnn.best_params_)

#Training for Valley
gridSearchKnn.fit(vTrainXScaled, vTrainY)
Vknn = KNeighborsRegressor()
Vknn.set_params(**gridSearchKnn.best_params_)

In [ ]:
#Random Forest regressor
paramGridRf = {
    "n_estimators": [50,500],
    "criterion": ['squared_error', 'absolute_error', 'poisson'],
    "max_features": ['sqrt', 'log2']
}
gridSearchRf = GridSearchCV(estimator=RandomForestRegressor(), param_grid=paramGridRf, cv=5, n_jobs=-1)

#Training for Aberporth
gridSearchRf.fit(aTrainX, aTrainY)
aRf = RandomForestRegressor()
aRf.set_params(**gridSearchRf.best_params_)

#Training for Cardiff
gridSearchRf.fit(cTrainX, cTrainY)
cRf = RandomForestRegressor()
cRf.set_params(**gridSearchRf.best_params_)

#Training for Valley
gridSearchRf.fit(vTrainX, vTrainY)
vRf = RandomForestRegressor()
vRf.set_params(**gridSearchRf.best_params_)

In [ ]:
#Support Vector Machine Regressor
paramGridSvm = {
    "kernel": ['rbf', 'linear', 'poly', 'sigmoid'],
    "C": [0.01, 10],
    "epsilon": [0.01, 10]
}
gridSearchSvm = GridSearchCV(estimator=SVR(), param_grid=paramGridSvm, cv=5, n_jobs=-1)

#Training for Aberporth
gridSearchSvm.fit(aTrainXScaled, aTrainY)
aSvr = SVR()
aSvr.set_params(**gridSearchSvm.best_params_)

#Training for Cardiff
gridSearchSvm.fit(cTrainXScaled, cTrainY)
cSvr = SVR()
cSvr.set_params(**gridSearchSvm.best_params_)

#Training for Valley
gridSearchSvm.fit(vTrainXScaled, vTrainY)
vSvr = SVR()
vSvr.set_params(**gridSearchSvm.best_params_)

Now all of the models have been trained for every single dataset. SVR and KNN datasets were scaled whereas Random forest wasn't.
It is also worth noting that RF took by far the longest time and data to train/optimise at 50.7s compared to KNN(10.2s) and SVM(1.9s)
There was a similar trend for the data usage as well, although I havent been able to pinpoint exact values atm.

An asterisk to put here is, that could be caused by the fact that the gridsearch in the RF code box simply had more things to run through and therefore took more time and memory. Will need further testing if I want to include it in the poster. 

Next we will predict using the test dataset and calculate which model does the best and by which metric and then come to a conclusion

In [ ]:
def getMetrics(preds, true):
    #Obtaining metrics
    accs = []
    precs = []
    recs = []

    for pred in preds:
        accs.append(true, pred)
        precs.append(true, pred)
        recs.append(true, pred, average='macro')

    return accs, precs, recs

In [ ]:
#Aberporth
aKnnPred = Aknn.predict(aTestXScaled)
aRfPred = aRf.predict(aTestX)
aSvrPred = aSvr.predict(aTestXScaled)
aPreds = [aKnnPred, aRfPred, aSvrPred]
Aaccs, Aprecs, Arecs = getMetrics(aPreds, aTestY)

#Cardiff
cKnnPred = Cknn.predict(cTestXScaled)
cRfPred = cRf.predict(cTestX)
cSvrPred = cSvr.predict(cTestXScaled)
cPreds = [cKnnPred, cRfPred, aSvrPred]
Caccs, Cprecs, Crecs = getMetrics(cPreds, cTestY)

#Valley
vKnnPred = Vknn.predict(vTestXScaled)
vRfPred = cRf.predict(cTestX)
vSvrPred = cSvr.predict(cTestXScaled)
vPreds = [vKnnPred, vRfPred, vSvrPred]
Vaccs, Vprecs, Vrecs = getMetrics(vPreds, vTestY)

In [ ]:
#Plotting
width = 0.4
plt.bar(x-width, Aaccs, width, color='cyan')
plt.bar(x, )